# NetFlow v1 CatBoost Model

`processed_netflow_v1` 산출물을 사용해 기존 모델과 분리된 `best_model_netflow_v1_catboost.cbm`을 학습한다.

In [1]:
from pathlib import Path
import json

import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training').exists():
    PROJECT_ROOT = Path.cwd().parents[1]

PROCESSED_DIR = PROJECT_ROOT / 'training/data/processed_netflow_v1'
MODEL_DIR = PROJECT_ROOT / 'training/artifacts/models'
MODEL_NAME = 'best_model_netflow_v1_catboost.cbm'

ITERATIONS = 800
LEARNING_RATE = 0.08
DEPTH = 8
RANDOM_STATE = 42

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR

PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/data/processed_netflow_v1')

In [2]:
x_train = np.load(PROCESSED_DIR / 'X_train.npy')
x_val = np.load(PROCESSED_DIR / 'X_val.npy')
x_test = np.load(PROCESSED_DIR / 'X_test.npy')
y_train = np.load(PROCESSED_DIR / 'y_train.npy')
y_val = np.load(PROCESSED_DIR / 'y_val.npy')
y_test = np.load(PROCESSED_DIR / 'y_test.npy')

label_mapping = json.loads((PROCESSED_DIR / 'label_mapping.json').read_text(encoding='utf-8'))
idx_to_label = {index: label for label, index in label_mapping.items()}
target_names = [idx_to_label[index] for index in sorted(idx_to_label)]

x_train.shape, x_val.shape, x_test.shape, target_names

((614453, 17),
 (131669, 17),
 (131669, 17),
 ['Benign',
  'Bot',
  'Brute Force',
  'DDoS',
  'DoS',
  'Infiltration',
  'SQL Injection'])

In [3]:
classes = np.array(sorted(idx_to_label), dtype=np.int64)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
dict(zip(target_names, weights.round(4)))

{'Benign': np.float64(0.627),
 'Bot': np.float64(7.9959),
 'Brute Force': np.float64(0.627),
 'DDoS': np.float64(0.627),
 'DoS': np.float64(0.627),
 'Infiltration': np.float64(2.0202),
 'SQL Injection': np.float64(3511.16)}

In [4]:
model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    iterations=ITERATIONS,
    learning_rate=LEARNING_RATE,
    depth=DEPTH,
    class_weights=weights.tolist(),
    random_seed=RANDOM_STATE,
    od_type='Iter',
    od_wait=50,
    verbose=50,
)
model.fit(x_train, y_train, eval_set=(x_val, y_val), use_best_model=True)

0:	learn: 0.7550636	test: 0.7594861	best: 0.7594861 (0)	total: 272ms	remaining: 3m 37s
50:	learn: 0.8679224	test: 0.8701462	best: 0.8701462 (50)	total: 10.9s	remaining: 2m 39s
100:	learn: 0.8712390	test: 0.8734198	best: 0.8734198 (100)	total: 20.9s	remaining: 2m 24s
150:	learn: 0.8728802	test: 0.8748564	best: 0.8748564 (150)	total: 30.4s	remaining: 2m 10s
200:	learn: 0.8745088	test: 0.8759032	best: 0.8759996 (194)	total: 40.1s	remaining: 1m 59s
250:	learn: 0.8755234	test: 0.8765045	best: 0.8765443 (248)	total: 50s	remaining: 1m 49s
300:	learn: 0.8761236	test: 0.8767837	best: 0.8767980 (298)	total: 59.8s	remaining: 1m 39s
350:	learn: 0.8766187	test: 0.8770152	best: 0.8770516 (341)	total: 1m 9s	remaining: 1m 28s
400:	learn: 0.8771179	test: 0.8772568	best: 0.8773585 (397)	total: 1m 19s	remaining: 1m 18s
450:	learn: 0.8776803	test: 0.8773966	best: 0.8774042 (438)	total: 1m 29s	remaining: 1m 8s
500:	learn: 0.8781991	test: 0.8549449	best: 0.8776053 (476)	total: 1m 39s	remaining: 59.2s
Stoppe

CatBoostClassifier(class_weights=[0.6269928571428571, 7.99590089269448, 0.6269928571428571, 0.6269928571428571, 0.6269928571428571, 2.0202301495972383, 3511.16], depth=8, eval_metric='TotalF1', iterations=800, learning_rate=0.08, loss_function='MultiClass', od_type='Iter', od_wait=50, random_seed=42, verbose=50)

In [5]:
model_path = MODEL_DIR / MODEL_NAME
model.save_model(model_path)

y_pred = model.predict(x_test).astype(np.int64).reshape(-1)
print(classification_report(
    y_test,
    y_pred,
    labels=classes,
    target_names=target_names,
    digits=4,
    zero_division=0,
))

report = classification_report(
    y_test,
    y_pred,
    labels=classes,
    target_names=target_names,
    digits=4,
    zero_division=0,
    output_dict=True,
)
matrix = confusion_matrix(y_test, y_pred, labels=classes).tolist()
meta = {
    'version': 'netflow_v1_catboost',
    'model': str(model_path),
    'processed_dir': str(PROCESSED_DIR),
    'num_features': int(x_train.shape[1]),
    'class_names': target_names,
    'label_mapping': label_mapping,
    'best_iteration': int(model.get_best_iteration() or 0),
    'params': {
        'iterations': ITERATIONS,
        'learning_rate': LEARNING_RATE,
        'depth': DEPTH,
        'random_state': RANDOM_STATE,
    },
    'classification_report': report,
    'confusion_matrix': matrix,
}
meta_path = MODEL_DIR / 'model_meta_netflow_v1.json'
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

model_path, meta_path

               precision    recall  f1-score   support

       Benign     0.9589    0.6419    0.7690     30000
          Bot     1.0000    1.0000    1.0000      2353
  Brute Force     0.7199    0.9998    0.8371     30000
         DDoS     0.9996    0.9996    0.9996     30000
          DoS     0.9998    0.6114    0.7588     30000
 Infiltration     0.4417    0.9114    0.5951      9311
SQL Injection     0.4545    1.0000    0.6250         5

     accuracy                         0.8235    131669
    macro avg     0.7964    0.8806    0.7978    131669
 weighted avg     0.8872    0.8235    0.8266    131669



(PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/artifacts/models/best_model_netflow_v1_catboost.cbm'),
 PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/artifacts/models/model_meta_netflow_v1.json'))